# 00. Data generation

procedural 생성 스크립트로 gt_corners.csv 변환부터 fake/similar/augmented 이미지 생성까지 단계별로 실행하고 확인한다.

- 1, 2절의 gt_corners.csv 변환과 보정 결과는 실제 `data/` 폴더가 아니라 `notebooks/data/public/` 아래에 쓴다.
- 3, 4, 5절의 fake, similar, augmented 이미지는 synthetic stage folder인 `data/synthetic/` 아래 `fake/`, `similar/`, `augmented/`에 category별 100개씩 생성하고 결과를 확인한다. 하위 category마다 `gt_corners.csv`를 따로 만든다.
- 7절의 measured 데이터는 `data/measured/`에 있는 기존 이미지와 LabelMe json을 그대로 gt_corners.csv로 변환하고 결과를 확인한다.
- 이 notebook은 agent가 실행하지 않는다. 모든 code cell은 사용자가 Jupyter에서 직접 실행한다.

## 0. 환경 설정

In [ ]:
# add project root to sys.path
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

In [ ]:
# import data generation and conversion modules
from types import SimpleNamespace

from src.data import smartdoc, midv2020, labelme
from data import fix_gt_corners
from data import make_augmented_images
from data import make_fake_images
from data import make_preview
from data import make_similar_images

In [ ]:
# define data directories and generation seed
NOTEBOOK_DATA_DIR = os.path.join(PROJECT_ROOT, "notebooks", "data")

# gt_corners.csv
PUBLIC_DIR = os.path.join(NOTEBOOK_DATA_DIR, "public")
SYNTHETIC_DIR = os.path.join(PROJECT_ROOT, "data", "synthetic")
FAKE_DIR = os.path.join(SYNTHETIC_DIR, "fake")
SIMILAR_DIR = os.path.join(SYNTHETIC_DIR, "similar")
AUGMENTED_DIR = os.path.join(SYNTHETIC_DIR, "augmented")

SMARTDOC_RAW_DIR = "E:/datasets/smartdoc_2015/smart_doc_extracted"
MIDV2020_RAW_DIR = "E:/datasets/midv_2020/midv2020_processed"

SEED = 42
MIN_DIST = 0.02
PREVIEW_COUNT = 50

print("notebook data dir:", NOTEBOOK_DATA_DIR)
print("synthetic stage dir:", SYNTHETIC_DIR)
print("smartdoc raw dir:", SMARTDOC_RAW_DIR)
print("midv2020 raw dir:", MIDV2020_RAW_DIR)

## 1. SMARTDOC 이미지 데이터셋 생성

SMARTDOC raw annotation을 gt_corners.csv로 변환하고 보정한 뒤 preview를 만든다.

- `src.data.smartdoc.create_data(data_dir, output_path)`로 원본 annotation을 `gt_corners.csv`로 변환한다. raw dataset은 project 밖의 `SMARTDOC_RAW_DIR`에 있고, 변환 결과는 실제 `data/public/`이 아니라 `notebooks/data/public/`에 쓴다.
- `data.fix_gt_corners`의 `fix_images`로 파일이 없는 row를 제거하고, `fix_corners`로 corner 순서를 TL, TR, BR, BL로 정렬하고 degenerate row를 제거한다. 두 함수 모두 `csv_path`를 그대로 덮어쓴다.
- `data.make_preview`로 `gt_corners.csv`와 같은 폴더에 `PREVIEW_COUNT`개의 preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# convert, fix, and preview smartdoc raw annotations into gt_corners.csv
smartdoc_output = os.path.join(PUBLIC_DIR, "smartdoc", "gt_corners.csv")
smartdoc_preview_dir = os.path.dirname(smartdoc_output)

if os.path.isdir(SMARTDOC_RAW_DIR):
    smartdoc.create_data(SMARTDOC_RAW_DIR, smartdoc_output)
    fix_gt_corners.fix_images(smartdoc_output)
    fix_gt_corners.fix_corners(smartdoc_output, min_dist=MIN_DIST)

    smartdoc_rows = make_preview.load_rows(smartdoc_output)
    smartdoc_preview_rows = make_preview.select_rows(
        smartdoc_rows,
        PREVIEW_COUNT,
        SEED,
    )
    make_preview.create_preview(
        smartdoc_preview_rows,
        smartdoc_preview_dir,
        PREVIEW_COUNT,
    )

    print("wrote", smartdoc_output)
    print("rows:", len(smartdoc_rows))
    for row in smartdoc_rows[:3]:
        print(row["image_dir"], row["image_name"])
else:
    print("skip: raw smartdoc dir not found:", SMARTDOC_RAW_DIR)

## 2. MIDV2020 이미지 데이터셋 생성

MIDV2020 raw annotation을 gt_corners.csv로 변환하고 보정한 뒤 preview를 만든다.

- `src.data.midv2020.create_data(data_dir, output_path)`로 원본 annotation을 `gt_corners.csv`로 변환한다. raw dataset은 project 밖의 `MIDV2020_RAW_DIR`에 있고, 변환 결과는 실제 `data/public/`이 아니라 `notebooks/data/public/`에 쓴다.
- `data.fix_gt_corners`의 `fix_images`로 파일이 없는 row를 제거하고, `fix_corners`로 corner 순서를 TL, TR, BR, BL로 정렬하고 degenerate row를 제거한다. 두 함수 모두 `csv_path`를 그대로 덮어쓴다.
- `data.make_preview`로 `gt_corners.csv`와 같은 폴더에 `PREVIEW_COUNT`개의 preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# convert, fix, and preview midv2020 raw annotations into gt_corners.csv
midv2020_output = os.path.join(PUBLIC_DIR, "midv2020", "gt_corners.csv")
midv2020_preview_dir = os.path.dirname(midv2020_output)

if os.path.isdir(MIDV2020_RAW_DIR):
    midv2020.create_data(MIDV2020_RAW_DIR, midv2020_output)
    fix_gt_corners.fix_images(midv2020_output)
    fix_gt_corners.fix_corners(midv2020_output, min_dist=MIN_DIST)

    midv2020_rows = make_preview.load_rows(midv2020_output)
    midv2020_preview_rows = make_preview.select_rows(
        midv2020_rows,
        PREVIEW_COUNT,
        SEED,
    )
    make_preview.create_preview(
        midv2020_preview_rows,
        midv2020_preview_dir,
        PREVIEW_COUNT,
    )

    print("wrote", midv2020_output)
    print("rows:", len(midv2020_rows))
    for row in midv2020_rows[:3]:
        print(row["image_dir"], row["image_name"])
else:
    print("skip: raw midv2020 dir not found:", MIDV2020_RAW_DIR)

## 3. Fake 이미지 생성과 확인

fake 이미지를 category별로 생성하고 gt_corners.csv 변환과 preview까지 확인한다.

- `data.make_fake_images`로 `fake_01`, `fake_02`, `fake_03` 3개 category의 fake 이미지를 `data/synthetic/fake/` 아래에 100개씩 생성한다. category folder가 이미 비어 있지 않으면 덮어쓰지 않고 건너뛴다.
- 생성 후 LabelMe annotation을 `gt_corners.csv`로 변환하고 corner를 보정한다.
- preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# generate fake images per category and build preview
fake_count = 100
fake_csv = os.path.join(FAKE_DIR, "gt_corners.csv")
fake_preview_dir = os.path.dirname(fake_csv)

fake_args = SimpleNamespace(
    count=fake_count,
    width=1920,
    height=1080,
    seed=SEED,
    background_dir=None,
)

make_fake_images.validate_args(fake_args)
fake_texture_paths = make_fake_images.list_backgrounds(
    fake_args.background_dir,
)
for geometry_profile, target_name in make_fake_images.PROFILE_TARGETS:
    category_dir = os.path.join(
        FAKE_DIR,
        "{}_{}".format(target_name, fake_count),
    )
    if os.path.isdir(category_dir) and os.listdir(category_dir):
        print("skip: existing category dir:", category_dir)
        continue
    make_fake_images.generate_dataset(
        geometry_profile,
        target_name,
        fake_args,
        fake_texture_paths,
    )

labelme.create_data(FAKE_DIR, fake_csv)
fix_gt_corners.fix_images(fake_csv)
fix_gt_corners.fix_corners(fake_csv, min_dist=MIN_DIST)

fake_rows = make_preview.load_rows(fake_csv)
fake_preview_rows = make_preview.select_rows(
    fake_rows,
    PREVIEW_COUNT,
    SEED,
)
make_preview.create_preview(
    fake_preview_rows,
    fake_preview_dir,
    PREVIEW_COUNT,
)

print("rows:", len(fake_rows))
for row in fake_rows[:3]:
    print(row["image_dir"], row["image_name"])

## 4. Similar 이미지 생성과 확인

similar 이미지를 category별로 생성하고 gt_corners.csv 변환과 preview까지 확인한다.

- `data.make_similar_images`로 `google`, `h8`, `q8`, `oppo` 4개 category의 similar 이미지를 `data/synthetic/similar/` 아래에 100개씩 생성한다. category folder가 이미 비어 있지 않으면 덮어쓰지 않고 건너뛴다.
- 생성 후 LabelMe annotation을 `gt_corners.csv`로 변환하고 corner를 보정한다.
- preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# generate similar images per category and build preview
similar_count = 100
similar_csv = os.path.join(SIMILAR_DIR, "gt_corners.csv")
similar_preview_dir = os.path.dirname(similar_csv)
similar_args = SimpleNamespace(
    count=similar_count,
    width=1920,
    height=1080,
    seed=SEED,
    output_root=SIMILAR_DIR,
    categories=make_similar_images.CATEGORY_ORDER,
)

make_similar_images.validate_args(similar_args)
for category in make_similar_images.CATEGORY_ORDER:
    category_dir = os.path.join(SIMILAR_DIR, "{}_{}".format(category, similar_count))
    if os.path.isdir(category_dir) and os.listdir(category_dir):
        print("skip: existing category dir:", category_dir)
        continue
    make_similar_images.generate_category(category, similar_args)

labelme.create_data(SIMILAR_DIR, similar_csv)
fix_gt_corners.fix_images(similar_csv)
fix_gt_corners.fix_corners(similar_csv, min_dist=MIN_DIST)

similar_rows = make_preview.load_rows(similar_csv)
similar_preview_rows = make_preview.select_rows(
    similar_rows,
    PREVIEW_COUNT,
    SEED,
)
make_preview.create_preview(
    similar_preview_rows,
    similar_preview_dir,
    PREVIEW_COUNT,
)

print("rows:", len(similar_rows))
for row in similar_rows[:3]:
    print(row["image_dir"], row["image_name"])

## 5. Augmented 이미지 생성과 확인

similar 이미지에 변형을 적용해 augmented 이미지를 생성하고 gt_corners.csv 변환과 preview까지 확인한다.

- `data.make_augmented_images`로 4절의 similar 이미지에 camera/photometric 변형을 적용해 `data/synthetic/augmented/` 아래에 category별 100개씩 생성한다. category folder가 이미 비어 있지 않으면 덮어쓰지 않고 건너뛴다.
- 생성 후 LabelMe annotation을 `gt_corners.csv`로 변환하고 corner를 보정한다.
- preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# generate augmented images from similar images and build preview
augmented_count = 100
augmented_csv = os.path.join(AUGMENTED_DIR, "gt_corners.csv")
augmented_preview_dir = os.path.dirname(augmented_csv)
augmented_args = SimpleNamespace(
    input_root=SIMILAR_DIR,
    output_root=AUGMENTED_DIR,
    categories=make_augmented_images.CATEGORY_ORDER,
    count=augmented_count,
    width=1920,
    height=1080,
    seed=SEED,
)

make_augmented_images.validate_args(augmented_args)
for category in make_augmented_images.CATEGORY_ORDER:
    category_dir = os.path.join(AUGMENTED_DIR, "{}_{}".format(category, augmented_count))
    if os.path.isdir(category_dir) and os.listdir(category_dir):
        print("skip: existing category dir:", category_dir)
        continue
    make_augmented_images.generate_category(category, augmented_args)

labelme.create_data(AUGMENTED_DIR, augmented_csv)
fix_gt_corners.fix_images(augmented_csv)
fix_gt_corners.fix_corners(augmented_csv, min_dist=MIN_DIST)

augmented_rows = make_preview.load_rows(augmented_csv)
augmented_preview_rows = make_preview.select_rows(
    augmented_rows,
    PREVIEW_COUNT,
    SEED,
)
make_preview.create_preview(
    augmented_preview_rows,
    augmented_preview_dir,
    PREVIEW_COUNT,
)

print("rows:", len(augmented_rows))
for row in augmented_rows[:3]:
    print(row["image_dir"], row["image_name"])

## 6. Measured 데이터셋 생성과 확인

measured raw image와 LabelMe annotation을 gt_corners.csv로 변환하고 보정한 뒤 preview를 만든다.

- `E:\fringe_data\test` 아래에는 이미지와 LabelMe로 작성한 json 파일만 있다. 별도 생성 단계 없이 `src.data.labelme.create_data`로 기존 파일을 `data/measured/test/gt_corners.csv`로 변환한다.
- `data.fix_gt_corners`의 `fix_images`로 파일이 없는 row를 제거하고, `fix_corners`로 corner 순서를 TL, TR, BR, BL로 정렬하고 degenerate row를 제거한다.
- preview 이미지를 만든 뒤 row 수와 예시를 확인한다.

In [ ]:
# convert, fix, and preview measured labelme annotations into gt_corners.csv
MEASURED_OUT_DIR = os.path.join(PROJECT_ROOT, "data", "measured", "test")
MEASURED_RAW_DIR = r"E:\fringe_data\test"
measured_csv = os.path.join(MEASURED_OUT_DIR, "gt_corners.csv")
measured_preview_dir = os.path.dirname(measured_csv)

if os.path.isdir(MEASURED_RAW_DIR):
    labelme.create_data(MEASURED_RAW_DIR, measured_csv)
    fix_gt_corners.fix_images(measured_csv)
    fix_gt_corners.fix_corners(measured_csv, min_dist=MIN_DIST)

    measured_rows = make_preview.load_rows(measured_csv)
    measured_preview_rows = make_preview.select_rows(
        measured_rows,
        PREVIEW_COUNT,
        SEED,
    )
    make_preview.create_preview(
        measured_preview_rows,
        measured_preview_dir,
        PREVIEW_COUNT,
    )

    print("wrote", measured_csv)
    print("rows:", len(measured_rows))
    for row in measured_rows[:3]:
        print(row["image_dir"], row["image_name"])
else:
    print("skip: measured raw dir not found:", MEASURED_RAW_DIR)

In [ ]:
# convert, fix, and preview measured labelme annotations into gt_corners.csv
MEASURED_OUT_DIR = os.path.join(PROJECT_ROOT, "data", "measured", "train")
MEASURED_RAW_DIR = r"E:\fringe_data\train"
measured_csv = os.path.join(MEASURED_OUT_DIR, "gt_corners.csv")
measured_preview_dir = os.path.dirname(measured_csv)

if os.path.isdir(MEASURED_RAW_DIR):
    labelme.create_data(MEASURED_RAW_DIR, measured_csv)
    fix_gt_corners.fix_images(measured_csv)
    fix_gt_corners.fix_corners(measured_csv, min_dist=MIN_DIST)

    measured_rows = make_preview.load_rows(measured_csv)
    measured_preview_rows = make_preview.select_rows(
        measured_rows,
        PREVIEW_COUNT,
        SEED,
    )
    make_preview.create_preview(
        measured_preview_rows,
        measured_preview_dir,
        PREVIEW_COUNT,
    )

    print("wrote", measured_csv)
    print("rows:", len(measured_rows))
    for row in measured_rows[:3]:
        print(row["image_dir"], row["image_name"])
else:
    print("skip: measured raw dir not found:", MEASURED_RAW_DIR)